# 2_main — Regression and Forecast

**Author:** Sota Yoshimoto  
Runs the regression models, validates them on 2024 data, and projects new
vehicle registrations and CO₂ emissions through 2040 for all 290 Swedish
municipalities. Reads the master panel produced by `1_API.ipynb`.

**Steps**
1. Load the master panel and run descriptive statistics
2. Fit three OLS specifications and compare adjusted R², MAE, NRMSE
3. Validate the best model on held-out 2024 data
4. Forecast new registrations 2025–2040 using CAGR projections of predictors
5. Combine with empirical scrapping rates, VKT and emission factors to
   project annual CO₂

Outputs (PNG plots, LaTeX table) are written to `./output/`.

**Result-dictionary ↔ paper-Model mapping**

| Variable | Paper |
| --- | --- |
| `results_summary_lin` | Linear baseline — diagnostic only (not in paper) |
| `results_summary_m1` | **Model 1** (log-log, 2023 cross-section) |
| `results_summary_m2` | **Model 2** (log-log, 2019–2023 panel) |
| `results_summary_m3` | **Model 3** (log-log + quadratic time trend, 2019–2023) |


## 1. Setup


In [ ]:
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ---------- Path configuration ----------
# Raw inputs live under DATA_DIR; outputs (CSV from 1_API, plots, LaTeX) go to OUTPUT_DIR.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Load the master panel produced by 1_API.ipynb. Force Kommun-kod and the
# 2-digit region code back to zero-padded strings to preserve leading zeros,
# and derive a one-letter Gruppkod_short (A/B/C) for any high-level grouping.
df = pd.read_csv(OUTPUT_DIR / "df.csv", dtype={"Kommuns-kod": "object"})
df['region_code'] = df['region_code'].astype(str).str.zfill(2)
df['Kommun-kod'] = df['Kommun-kod'].astype(str).str.zfill(4)
df['Gruppkod_short'] = df['Gruppkod'].str[:1]
df["Municipality"] = df["Municipality"].str.rstrip()


## 2. Descriptive analysis


In [ ]:
# Define the five modelled fuel types plus Total, and the six continuous
# socioeconomic predictors that enter every regression specification.
fuel_lists = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids", "Total"]
candidate_vars = [
    "median_household_income",
    "bachelor_plus_pct",   # share, 0–100
    "household_num", 
    "household_num_growth",   # share, 0–100
    "Avg_Price_öre_kWh",
    "corporation_percentage"
]

In [ ]:
# Build the descriptive-statistics table (Table 4.1 in the paper):
# mean / std / min / max for new registrations by fuel and for the six
# explanatory variables, all measured in 2023 across the 290 kommuner.
# 1. Prepare the full list of columns to describe
# Map fuel names to the actual column names used in the model (e.g., new_Petrol)
target_fuel_cols = ["new_" + fuel for fuel in fuel_lists]
all_cols_to_describe = target_fuel_cols + candidate_vars

# 2. Filter data for the year 2023
df_2023 = df[df["year"] == 2023].copy()

# 3. Calculate descriptive statistics and filter for specific metrics
# We transpose (.T) at the end to make it easier to read (variables as rows)
descriptive_stats = df_2023[all_cols_to_describe].describe().loc[['mean', 'std', 'min', 'max']].T

# 4. Display the result
print("=== Descriptive Statistics for 2023 ===")
display(descriptive_stats.round(4))

### 2.1 Growth-rate bar chart by SKR group


In [ ]:
# Average income- and household-growth rate by SKR group. Useful as a
# sanity check that the panel-level growth signal is non-trivial across
# groups before they enter the regression as a categorical control.
import matplotlib.pyplot as plt
df_grouped = df.groupby("Gruppkod")[['median_household_income_growth','household_num_growth']].mean()
df_grouped.plot(kind="bar", figsize=(12,6), width=0.8)

plt.title("Growth Metrics by Kommun Group (SKR Classification)")
plt.ylabel("Growth Rate")
plt.xlabel("Kommun Group")
plt.xticks(rotation=45)
plt.legend(["Income Growth", "Household Growth"])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# PLOT: TOTAL NEW REGISTRATIONS BY GRUPPKOD (2019-2024)
# ==========================================

# 1. Configuration
fuel_lists = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]
target_cols = [f"new_{f}" for f in fuel_lists]

colors = {
    "Petrol": "red", 
    "Diesel": "black", 
    "Electricity": "green", 
    "Electric-hybrids": "purple", 
    "Plug-in-hybrids": "blue"
}

# 2. Aggregation: sum registrations per Gruppkod and year
df_group_sum = (
    df[df["year"].between(2019, 2024)]
    .groupby(["Gruppkod", "year"])[target_cols]
    .sum()
    .reset_index()
)

# 3. Grid setup
groups = sorted(df_group_sum["Gruppkod"].unique())  # ['A1','A2','B3','B4','B5','C6','C7','C8','C9']
n_cols = 3
n_rows = (len(groups) + n_cols - 1) // n_cols

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows), sharex=True)
axes = axes.flatten()

# 4. Plot each Gruppkod
for i, group in enumerate(groups):
    ax = axes[i]
    group_df = df_group_sum[df_group_sum["Gruppkod"] == group].sort_values("year")
    
    for fuel in fuel_lists:
        col = f"new_{fuel}"
        ax.plot(
            group_df["year"],
            group_df[col],
            marker="o",
            linewidth=2,
            label=fuel,
            color=colors[fuel]
        )
    
    ax.set_title(f"Group {group}", fontsize=14, fontweight="bold")
    ax.set_ylabel("Total New Registrations")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.set_xticks(range(2019, 2025))
    
    if i == 0:
        ax.legend(title="Fuel Type", fontsize="small")

# Remove unused subplots
for j in range(len(groups), len(axes)):
    fig.delaxes(axes[j])

# 5. Global formatting
fig.supxlabel("Year", fontsize=13)
plt.suptitle(
    "Total New Registrations by Fuel Type per Gruppkod (2019–2024)",
    fontsize=18, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# PLOT: NEW REGISTRATIONS BY FUEL TYPE
# Layout: 2 rows x 3 cols, legend in cell (2,3)
# ==========================================

# 1. Configuration
fuel_lists = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]
target_cols = [f"new_{f}" for f in fuel_lists]

# Group colors
group_colors = {
    "A1": "#e41a1c",
    "A2": "#ff7f00",
    "B3": "#daa520",
    "B4": "#4daf4a",
    "B5": "#00ced1",
    "C6": "#377eb8",
    "C7": "#984ea3",
    "C8": "#a65628",
    "C9": "#999999"
}

# English labels from SKR table
group_labels = {
    "A1": "A1: Large cities",
    "A2": "A2: Commuter municipalities (large city)",
    "B3": "B3: Larger cities",
    "B4": "B4: Commuter municipalities (larger city)",
    "B5": "B5: Low commuting",
    "C6": "C6: Smaller cities",
    "C7": "C7: Commuter municipalities (smaller city)",
    "C8": "C8: Rural municipalities",
    "C9": "C9: Rural (Tourism)"
}

# 2. Aggregation
df_group_sum = (
    df[df["year"].between(2019, 2024)]
    .groupby(["Gruppkod", "year"])[target_cols]
    .sum()
    .reset_index()
)

groups = sorted(df_group_sum["Gruppkod"].unique())

# 3. Figure: 2 rows x 3 cols
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes_flat = axes.flatten()

line_handles = []

for i, fuel in enumerate(fuel_lists):
    ax = axes_flat[i]
    col = f"new_{fuel}"
    handles = []

    for group in groups:
        group_df = df_group_sum[df_group_sum["Gruppkod"] == group].sort_values("year")
        line, = ax.plot(
            group_df["year"],
            group_df[col],
            marker="o",
            linewidth=2,
            label=group_labels[group],
            color=group_colors[group]
        )
        handles.append(line)

    if i == 0:
        line_handles = handles  # Save for legend panel

    ax.set_title(fuel, fontsize=13, fontweight="bold")
    ax.set_ylabel("Total New Registrations")
    ax.set_xticks(range(2019, 2025))
    ax.grid(True, linestyle=":", alpha=0.6)

# 4. Legend panel in cell [1, 2] (bottom-right)
ax_legend = axes_flat[5]
ax_legend.axis("off")
ax_legend.legend(
    handles=line_handles,
    labels=[group_labels[g] for g in groups],
    title="Gruppkod (SKR Classification)",
    title_fontsize=11,
    fontsize=10,
    loc="center",
    frameon=True,
    framealpha=0.9
)

# 5. Global formatting
fig.supxlabel("Year", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Regression

Three specifications are fit on log-transformed new registrations (paper
Section 3.3). A linear-scale baseline is also fit purely to motivate the
log transformation via residual diagnostics.


In [ ]:
# Linear-scale baseline regression on 2023 data. NOT used as a paper
# model: this exists only to demonstrate, via the residual plots below,
# that homoscedasticity fails on raw registration counts and motivates
# the log transformation (paper Section 3.4).
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Set timeframe for Model 2 (2023 only)
START_YEAR = 2023
END_YEAR = 2023
REFERENCE_GROUP = "Gruppkod_B3"

# 2. Filter data
df_filtered = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)].copy()

# 3. Create Dummy Variables for Gruppkod
X_categorical = pd.get_dummies(df_filtered["Gruppkod"], prefix="Gruppkod", dtype=float)

# 4. Merge back to the filtered dataframe
# reset_index(drop=True) is crucial to align the rows after concatenation
df_model_ready = pd.concat([df_filtered.reset_index(drop=True), X_categorical.reset_index(drop=True)], axis=1)

# 5. Define fixed variables (Excluding the Reference Group)
fixed_vars = [c for c in X_categorical.columns if c != REFERENCE_GROUP]

def get_sig_stars(p):
    """Returns academic significance stars for p-values."""
    if p < 0.01: return "***"
    elif p < 0.05: return "**"
    elif p < 0.1: return "*"
    return ""

# ==========================================
# 1. EXECUTION ENGINE: LINEAR OLS (MODEL 1)
# ==========================================
results_summary_lin = {}
fuel_lists = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids", "Total"]

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_model_ready.columns: continue

    # Data Cleaning for the specific fuel type
    current_cols = [y_col] + candidate_vars + fixed_vars
    df_clean = df_model_ready[current_cols].apply(pd.to_numeric, errors='coerce').dropna()
    
    y = df_clean[y_col]
    X_cand = df_clean[candidate_vars]
    X_fixed = df_clean[fixed_vars]
    X_matrix = sm.add_constant(pd.concat([X_fixed, X_cand], axis=1))

    # Fit Linear Model (Level-Level)
    model_lin = sm.OLS(y, X_matrix).fit()

    # --- Metrics Calculation ---
    y_true = y
    # Car registrations cannot be negative in the real world
    y_pred_raw = model_lin.predict(X_matrix)
    y_pred_clipped = y_pred_raw.clip(lower=0) 

    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)
    
    # Normalized RMSE (Max-Min)
    y_range = y_true.max() - y_true.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan

    # Store results for comparison matrix later
    results_summary_lin[fuel] = {
        "model": model_lin,
        "adj_r2": model_lin.rsquared_adj,
        "rmse": rmse,
        "mae": mae,
        "nrmse": nrmse,
        "nobs": model_lin.nobs,
        "fitted": y_pred_raw,
        "resid": model_lin.resid
    }

# ==========================================
# 2. TABLE GENERATOR: MODEL 1 RESULTS
# ==========================================
table_output_lin = {}
row_order_lin = ["const"] + candidate_vars + fixed_vars

for fuel in fuel_lists:
    if fuel not in results_summary_lin: continue
    res = results_summary_lin[fuel]
    m = res["model"]
    
    formatted_col = {}
    for var in row_order_lin:
        if var in m.params:
            val, se, p = m.params[var], m.bse[var], m.pvalues[var]
            formatted_col[var] = f"{val:.3f}{get_sig_stars(p)} ({se:.3f})"
        else:
            formatted_col[var] = "---"
    
    formatted_col['---'] = "---"
    formatted_col['N'] = str(int(res["nobs"]))
    formatted_col['Adj. R-squared'] = f"{res['adj_r2']:.3f}"
    formatted_col['RMSE'] = f"{res['rmse']:.2f}"
    formatted_col['NRMSE (Max-Min)'] = f"{res['nrmse']:.4f}"
    formatted_col['MAE'] = f"{res['mae']:.2f}"
    table_output_lin[fuel] = pd.Series(formatted_col)

# Display Final Table
final_df_lin = pd.DataFrame(table_output_lin)
clean_index_lin = {
    "const": "Constant",
    "median_household_income": "Median Household Income",
    "bachelor_plus_pct": "Higher Education (%)",
    "household_num": "Total Households",
    "household_num_growth": "Household Growth",
    "deg_urbanization": "Urbanization Degree",
    "Avg_Price_öre_kWh": "Electricity Price",
    "corporation_percentage": "Corporate Share (%)"
}
final_df_lin.rename(index=clean_index_lin, inplace=True)
print("=== LINEAR BASELINE (diagnostic only): LINEAR REGRESSION (2023) ===")
display(final_df_lin)

### 3.1 Model 1 — Log-log specification on the 2023 cross-section


In [ ]:
# Paper Model 1: log(1+y) regressed on socioeconomic predictors and
# Gruppkod dummies (B3 as reference). The household-count predictor is
# log-transformed; metrics are reported back on the real (un-logged) scale.
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ==========================================
# 2-1. CONFIGURATION & LOG-SETTING
# ==========================================
# (START_YEAR, END_YEAR, REFERENCE_GROUP, candidate_vars are reused from the preceding cells)

# Variables to log-transform (Independent)
vars_to_log_x = ["household_num"]

# ==========================================
# 2. EXECUTION ENGINE (LOG-TRANSFORMS & METRICS with NRMSE)
# ==========================================
results_summary_m1 = {}

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_model_ready.columns: continue

    current_cols = [y_col] + candidate_vars + fixed_vars
    df_clean = df_model_ready[current_cols].apply(pd.to_numeric, errors='coerce').dropna()
    
    y_transformed = np.log1p(df_clean[y_col])
    X_cand = df_clean[candidate_vars].copy()
    for var in vars_to_log_x:
        X_cand[var] = np.log1p(X_cand[var])
        X_cand = X_cand.rename(columns={var: f"log_{var}"})
    
    X_fixed = df_clean[fixed_vars]
    X_full = sm.add_constant(pd.concat([X_fixed, X_cand], axis=1))

    model_full = sm.OLS(y_transformed, X_full).fit()

    # --- CALCULATE METRICS IN REAL SCALE ---
    y_true_real = df_clean[y_col]
    y_pred_real = np.expm1(model_full.predict(X_full))

    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    
    # --- NRMSE (Max-Min) Calculation ---
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan

    results_summary_m1[fuel] = {
        "model": model_full,
        "adj_r2": model_full.rsquared_adj,
        "rmse": rmse,
        "mae": mae,
        "nrmse": nrmse,
        "nobs": model_full.nobs
    }

clean_index_map = {
    "const": "Constant",
    "median_household_income": "Median Household Income",
    "bachelor_plus_pct": "Higher Education (%)",
    "household_num": "Log(Total Households)",
    "household_num_growth": "Household Growth",
    "Avg_Price_öre_kWh": "Electricity Price",
    "corporation_percentage": "Corporate Share (%)"
}

# ==========================================
# 2-3. TABLE GENERATOR (Updated with NRMSE)
# ==========================================
table_output_log = {}
row_order_log = ["const"] + [f"log_{v}" if v in vars_to_log_x else v for v in candidate_vars] + fixed_vars

for fuel in fuel_lists:
    if fuel not in results_summary_m1: continue
    m = results_summary_m1[fuel]["model"]
    res = results_summary_m1[fuel]
    
    formatted_col = {}
    for var in row_order_log:
        if var in m.params:
            val, se, p = m.params[var], m.bse[var], m.pvalues[var]
            formatted_col[var] = f"{val:.3f}{get_sig_stars(p)} ({se:.3f})"
        else:
            formatted_col[var] = "---"
    
    formatted_col['---'] = "---"
    formatted_col['N'] = str(int(res["nobs"]))
    formatted_col['Adj. R-squared'] = f"{res['adj_r2']:.3f}"
    formatted_col['RMSE (Real Scale)'] = f"{res['rmse']:.2f}"
    formatted_col['NRMSE (Max-Min)'] = f"{res['nrmse']:.4f}"
    formatted_col['MAE (Real Scale)'] = f"{res['mae']:.2f}"
    table_output_log[fuel] = pd.Series(formatted_col)

final_df_log = pd.DataFrame(table_output_log)
final_df_log.rename(index=clean_index_map, inplace=True)
print("=== MODEL 1: LOG-LOG REGRESSION RESULTS (2023) ===")
display(final_df_log)

### 3.2 Residual analysis

Side-by-side residuals from the linear baseline vs the log-log Model 1
specification (paper Figure 3.2). The funnel-shaped residuals on the
linear scale collapse to a near-uniform spread on the log scale,
supporting the log transformation.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ==========================================
# RESIDUAL DIAGNOSTICS: Before vs After Log
# 6 rows × 2 cols, one row per fuel type: left = linear, right = log scale
# ==========================================

fig, axes = plt.subplots(6, 2, figsize=(14, 30))
#fig.suptitle("Residuals vs Fitted: Before vs After Log Transformation",
#            fontsize=15, fontweight="bold")

for i, fuel in enumerate(fuel_lists):

    # --- Left column: before log transform (results_summary_lin) ---
    ax_lin = axes[i, 0]
    if fuel in results_summary_lin:
        res = results_summary_lin[fuel]
        sns.scatterplot(x=res["fitted"], y=res["resid"], ax=ax_lin, alpha=0.5)
        ax_lin.axhline(y=0, color='red', linestyle='--')
        ax_lin.set_title(f'{fuel}: Residuals vs Fitted (Linear)')
        ax_lin.set_xlabel('Predicted Registrations')
        ax_lin.set_ylabel('Residuals')

    # --- Right column: after log transform (results_summary_m1) ---
    ax_log = axes[i, 1]
    if fuel in results_summary_m1:
        res = results_summary_m1[fuel]
        m = res["model"]
        fitted_vals = m.fittedvalues
        residuals   = m.resid
        sns.scatterplot(x=fitted_vals, y=residuals, ax=ax_log, alpha=0.5, color='teal')
        ax_log.axhline(y=0, color='red', linestyle='--')
        ax_log.set_title(f'{fuel}: Residuals vs Fitted (Log Scale)')
        ax_log.set_xlabel('Predicted Log(Registrations)')
        ax_log.set_ylabel('Log Residuals')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "residual_before_after_log.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fuel_splits = [fuel_lists[:3], fuel_lists[3:6]]  # first three, then last three

for fig_num, fuels_subset in enumerate(fuel_splits, start=1):
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    
    for i, fuel in enumerate(fuels_subset):
        
        # --- Left column: linear-scale residuals ---
        ax_lin = axes[i, 0]
        if fuel in results_summary_lin:
            res = results_summary_lin[fuel]
            sns.scatterplot(x=res["fitted"], y=res["resid"], ax=ax_lin, alpha=0.5)
            ax_lin.axhline(y=0, color='red', linestyle='--')
            ax_lin.set_title(f'{fuel}: Residuals vs Fitted (Linear)')
            ax_lin.set_xlabel('Predicted Registrations')
            ax_lin.set_ylabel('Residuals')
        
        # --- Right column: log-scale residuals ---
        ax_log = axes[i, 1]
        if fuel in results_summary_m1:
            res = results_summary_m1[fuel]
            m = res["model"]
            sns.scatterplot(x=m.fittedvalues, y=m.resid, ax=ax_log, alpha=0.5, color='teal')
            ax_log.axhline(y=0, color='red', linestyle='--')
            ax_log.set_title(f'{fuel}: Residuals vs Fitted (Log Scale)')
            ax_log.set_xlabel('Predicted Log(Registrations)')
            ax_log.set_ylabel('Log Residuals')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"residual_before_after_log_{fig_num}.png", dpi=300, bbox_inches="tight")
    plt.show()

### 3.3 Model 2 — Log-log on the 2019–2023 panel (no trend)


In [ ]:
# Paper Model 2: same specification as Model 1, but estimated on the full
# 2019–2023 panel without a time trend. Tests whether broadening the
# estimation window without trend terms improves predictive accuracy.
# ==========================================
# 3-1. CONFIGURATION (FULL PERIOD)
# ==========================================
START_YEAR_FULL = 2019
END_YEAR_FULL = 2023
# vars_to_log_x, candidate_vars, fuel_lists are inherited from previous cells

# ==========================================
# 3-2. PREPROCESSING PIPELINE
# ==========================================
df_filtered_full = df[(df["year"] >= START_YEAR_FULL) & (df["year"] <= END_YEAR_FULL)].copy()

# Universal Dummy Creation for full period
X_cat_full = pd.get_dummies(df_filtered_full["Gruppkod"], prefix="Gruppkod", dtype=float)
df_model_ready_full = pd.concat([df_filtered_full.reset_index(drop=True), X_cat_full.reset_index(drop=True)], axis=1)
fixed_vars_full = [c for c in X_cat_full.columns if c != REFERENCE_GROUP]

# ==========================================
# 3-3. EXECUTION ENGINE (LOG-TRANSFORMS & METRICS with NRMSE)
# ==========================================
results_summary_m2 = {}

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_model_ready_full.columns: continue

    current_cols = [y_col] + candidate_vars + fixed_vars_full
    df_clean = df_model_ready_full[current_cols].apply(pd.to_numeric, errors='coerce').dropna()
    
    # --- LOG TRANSFORMATIONS ---
    y_transformed = np.log1p(df_clean[y_col])
    X_cand = df_clean[candidate_vars].copy()
    for var in vars_to_log_x:
        X_cand[var] = np.log1p(X_cand[var])
        X_cand = X_cand.rename(columns={var: f"log_{var}"})
    
    X_fixed = df_clean[fixed_vars_full]
    X_matrix = sm.add_constant(pd.concat([X_fixed, X_cand], axis=1))

    # Fit Full Model
    model_full = sm.OLS(y_transformed, X_matrix).fit()

    # --- CALCULATE METRICS IN REAL SCALE ---
    y_true_real = df_clean[y_col]
    y_pred_real = np.expm1(model_full.predict(X_matrix))

    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    
    # --- NRMSE (Max-Min) ---
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan

    results_summary_m2[fuel] = {
        "model": model_full,
        "adj_r2": model_full.rsquared_adj,
        "rmse": rmse,
        "mae": mae,
        "nrmse": nrmse, # Added
        "aic": model_full.aic,
        "nobs": model_full.nobs
    }

# ==========================================
# 3-4. UNIFIED TABLE GENERATOR (Updated with NRMSE)
# ==========================================
table_output_log_full = {}
row_order_log = ["const"] + [f"log_{v}" if v in vars_to_log_x else v for v in candidate_vars] + fixed_vars_full

for fuel in fuel_lists:
    if fuel not in results_summary_m2: continue
    res = results_summary_m2[fuel]
    m = res["model"]
    
    formatted_col = {}
    for var in row_order_log:
        if var in m.params:
            val, se, p = m.params[var], m.bse[var], m.pvalues[var]
            formatted_col[var] = f"{val:.3f}{get_sig_stars(p)} ({se:.3f})"
        else:
            formatted_col[var] = "---"
    
    formatted_col['---'] = "---"
    formatted_col['N'] = str(int(res["nobs"]))
    formatted_col['Adj. R-squared'] = f"{res['adj_r2']:.3f}"
    formatted_col['RMSE (Real)'] = f"{res['rmse']:.2f}"
    formatted_col['NRMSE (Max-Min)'] = f"{res['nrmse']:.4f}" # Added
    formatted_col['MAE (Real)'] = f"{res['mae']:.2f}"
    
    table_output_log_full[fuel] = pd.Series(formatted_col)

final_df_log_full = pd.DataFrame(table_output_log_full)
final_df_log_full.rename(index=clean_index_map, inplace=True)
print(f"=== MODEL 2: LOG-LOG REGRESSION SUMMARY ({START_YEAR_FULL}-{END_YEAR_FULL}) ===")
display(final_df_log_full)



### 3.4 Model 3 — Log-log with quadratic time trend (2019–2023)


In [ ]:
# Paper Model 3: same as Model 2 plus a linear trend (tau = year - 2019)
# and its square. These pick up non-linear momentum in the energy
# transition that the socioeconomic predictors alone do not capture
# (paper Section 5.3).
# ==========================================
# 5-1. PREPROCESSING & TREND SETTING
# ==========================================
df_full_trend = df[df["year"].between(2019, 2023)].copy()

# Trend variable indexed to t=0 in 2019
df_full_trend['trend'] = df_full_trend['year'] - 2019
df_full_trend['trend_sq'] = df_full_trend['trend'] ** 2

# Create Gruppkod dummy variables
X_cat_trend = pd.get_dummies(df_full_trend["Gruppkod"], prefix="Gruppkod", dtype=float)
df_model_ready_trend = pd.concat([df_full_trend.reset_index(drop=True), X_cat_trend.reset_index(drop=True)], axis=1)
fixed_vars_trend = [c for c in X_cat_trend.columns if c != REFERENCE_GROUP]

# ==========================================
# 5-2. EXECUTION ENGINE (LOG-LOG + TREND with NRMSE)
# ==========================================
results_summary_m3 = {}

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_model_ready_trend.columns: continue

    current_cols = [y_col] + candidate_vars + fixed_vars_trend + ['trend', 'trend_sq']
    df_clean = df_model_ready_trend[current_cols].apply(pd.to_numeric, errors='coerce').dropna()
    
    # --- TRANSFORMATIONS ---
    y_log = np.log1p(df_clean[y_col])
    X_cand = df_clean[candidate_vars].copy()
    if "household_num" in X_cand.columns:
        X_cand["household_num"] = np.log1p(X_cand["household_num"])
        X_cand = X_cand.rename(columns={"household_num": "log_household_num"})
    
    X_fixed = df_clean[fixed_vars_trend]
    X_time = df_clean[['trend', 'trend_sq']]
    X_matrix = sm.add_constant(pd.concat([X_fixed, X_cand, X_time], axis=1))

    model_trend = sm.OLS(y_log, X_matrix).fit()

    # --- CALCULATE METRICS IN REAL SCALE ---
    y_true_real = df_clean[y_col]
    y_pred_real = np.expm1(model_trend.predict(X_matrix))

    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    
    # --- NRMSE (Max-Min) ---
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan

    results_summary_m3[fuel] = {
        "model": model_trend,
        "adj_r2": model_trend.rsquared_adj,
        "rmse": rmse,
        "mae": mae,
        "nrmse": nrmse, # Added
        "nobs": model_trend.nobs
    }

# ==========================================
# 5-3. UNIFIED TABLE GENERATOR (Updated with NRMSE)
# ==========================================
table_output_trend = {}
row_order_trend = ["const"] + [f"log_{v}" if v in vars_to_log_x else v for v in candidate_vars] + ['trend', 'trend_sq'] + fixed_vars_trend

for fuel in fuel_lists:
    if fuel not in results_summary_m3: continue
    res = results_summary_m3[fuel]
    m = res["model"]
    
    formatted_col = {}
    for var in row_order_trend:
        if var in m.params:
            val, se, p = m.params[var], m.bse[var], m.pvalues[var]
            formatted_col[var] = f"{val:.3f}{get_sig_stars(p)} ({se:.3f})"
        else:
            formatted_col[var] = "---"
    
    formatted_col['---'] = "---"
    formatted_col['N'] = str(int(res["nobs"]))
    formatted_col['Adj. R-squared'] = f"{res['adj_r2']:.3f}"
    formatted_col['RMSE (Real)'] = f"{res['rmse']:.2f}"
    formatted_col['NRMSE (Max-Min)'] = f"{res['nrmse']:.4f}" # Added
    formatted_col['MAE (Real)'] = f"{res['mae']:.2f}"
    
    table_output_trend[fuel] = pd.Series(formatted_col)

clean_index_trend = {
    "const": "Constant",
    "median_household_income": "Median Household Income",
    "bachelor_plus_pct": "Higher Education (%)",
    "household_num": "Log(Total Households)",
    "household_num_growth": "Household Growth",
    "Avg_Price_öre_kWh": "Electricity Price",
    "corporation_percentage": "Corporate Share (%)",
    "trend": "Trend",
    "trend_sq": "Trend Squared"
}

final_df_trend = pd.DataFrame(table_output_trend)
final_df_trend.rename(index=clean_index_trend, inplace=True)
print("=== MODEL 3: LOG-LOG WITH QUADRATIC TREND RESULTS ===")
display(final_df_trend)

### 3.5 Comparison matrix


In [ ]:
# Side-by-side comparison of Adj. R^2, MAE and NRMSE across all three
# paper models. Reproduces the in-sample portion of paper Table 4.2.
import pandas as pd
import numpy as np

# ==========================================
# FINAL CONSOLIDATED MODEL COMPARISON TABLE
# ==========================================

all_metrics = []

# Define standard lists for ordering
fuel_lists = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids", "Total"]
metrics_order = ["Adj. R2", "MAE", "NRMSE"]
models_order = ["Model (2023)", "Model (2019-23)", "Model (2019-23+Trend)"]

# Map our result dictionaries to the new model names
models_info = [
    ("Model (2023)", results_summary_m1, [2023]),
    ("Model (2019-23)", results_summary_m2, range(2019, 2023)),
    ("Model (2019-23+Trend)", results_summary_m3, range(2019, 2023))
]

for model_name, results_dict, years in models_info:
    for fuel in fuel_lists:
        if fuel not in results_dict: continue
        
        res = results_dict[fuel]
        
        # Calculate NRMSE based on the data range of each specific model
        actual_y = df[df["year"].isin(years)][f"new_{fuel}"]
        y_range = actual_y.max() - actual_y.min()
        nrmse = res["rmse"] / y_range if y_range > 0 else np.nan

        all_metrics.append({
            "Fuel": fuel,
            "Model": model_name,
            "Adj. R2": res["adj_r2"],
            "MAE": res["mae"],
            "NRMSE": nrmse
        })

# 1. Create DataFrame
df_master = pd.DataFrame(all_metrics)

# 2. Pivot Table
df_master_pivot = df_master.pivot(index="Fuel", columns="Model")

# 3. Reorder Rows (Fuels)
df_master_pivot = df_master_pivot.reindex(fuel_lists)

# 4. Reorder Columns (Multi-index)
# Level 0 is Metrics (Adj. R2, MAE, NRMSE)
# Level 1 is Models (Model (2023), etc.)
df_master_pivot = df_master_pivot.reindex(columns=metrics_order, level=0)
df_master_pivot = df_master_pivot.reindex(columns=models_order, level=1)

print("=== FINAL CONSOLIDATED COMPARISON MATRIX ===")
print("Rows: Sorted by fuel_lists")
print("Columns: Sorted by metrics and model versions")
display(df_master_pivot.round(3))

## 4. Out-of-sample validation on 2024

Each fitted model from Section 3 is asked to predict 2024 new registrations
(the year held out from estimation). Performance is reported as test R²,
RMSE, NRMSE and MAE on the real (un-logged) scale.


### 4.1 Validating Model 1 (trained on 2023) on 2024


In [ ]:
# Predict 2024 with Model 1 (trained on 2023 only). The 2024 feature
# matrix is built exactly as during training: same dummy alignment,
# same log transform of household count, and the predicted log target
# is back-transformed with expm1 and clipped at zero.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. PREPARE 2024 TEST DATA
# ==========================================
TEST_YEAR = 2024

# Filter data for 2024
df_2024 = df[df["year"] == TEST_YEAR].copy()

# Create Dummy Variables for 2024
X_cat_2024 = pd.get_dummies(df_2024["Gruppkod"], prefix="Gruppkod", dtype=float)

# Align Dummies: Ensure 2024 has the same columns as the training set (fixed_vars)
for col in fixed_vars:
    if col not in X_cat_2024.columns:
        X_cat_2024[col] = 0.0

# Merge back and clean
df_test_2024 = pd.concat([df_2024.reset_index(drop=True), X_cat_2024.reset_index(drop=True)], axis=1)

# ==========================================
# 2. RUN PREDICTION & EVALUATION
# ==========================================
test_metrics_2024 = []

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_test_2024.columns or fuel not in results_summary_m1:
        continue
    
    # Retrieve the trained 2023 model
    m_info = results_summary_m1[fuel]
    model_2023 = m_info["model"]
    
    # Prepare 2024 features (Apply same transformations as training)
    df_clean_2024 = df_test_2024[[y_col] + candidate_vars + fixed_vars].dropna()
    
    X_test = df_clean_2024[candidate_vars].copy()
    # Log transform households as done in Model 1
    if "household_num" in X_test.columns:
        X_test["household_num"] = np.log1p(X_test["household_num"])
        X_test = X_test.rename(columns={"household_num": "log_household_num"})
    
    X_matrix_2024 = sm.add_constant(pd.concat([df_clean_2024[fixed_vars], X_test], axis=1))
    # Ensure column order matches the model parameters
    X_matrix_2024 = X_matrix_2024[model_2023.params.index]

    # --- PREDICT ---
    # Model predicts log(y+1), so we back-transform with expm1
    y_pred_log = model_2023.predict(X_matrix_2024)
    y_pred_real = np.expm1(y_pred_log).clip(lower=0)
    y_true_real = df_clean_2024[y_col]

    # --- CALCULATE METRICS ---
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    
    # NRMSE (Max-Min)
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan
    
    # Test R-squared (How much of 2024 variance the 2023 model explains)
    r2_test = r2_score(y_true_real, y_pred_real)

    test_metrics_2024.append({
        "Fuel": fuel,
        "Test R2 (2024)": r2_test,
        "RMSE": rmse,
        "NRMSE": nrmse,
        "MAE": mae
    })

# ==========================================
# 3. DISPLAY VALIDATION REPORT
# ==========================================
df_validation_2024 = pd.DataFrame(test_metrics_2024).set_index("Fuel")

print(f"=== VALIDATION REPORT: Model (2023) tested on {TEST_YEAR} Data ===")
display(df_validation_2024.round(3))

### 4.2 Validating Model 2 (trained on 2019–2023, no trend) on 2024


In [ ]:
# ==========================================
# VALIDATION: Model (2019-23) tested on 2024
# ==========================================
test_metrics_m3 = []

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    if y_col not in df_test_2024.columns or fuel not in results_summary_m2:
        continue
    
    # 1. Retrieve the trained model (paper Model 2)
    res = results_summary_m2[fuel]
    model_m3 = res["model"]
    
    # 2. Prepare 2024 Features
    df_clean_2024 = df_test_2024[[y_col] + candidate_vars + fixed_vars].dropna()
    X_test = df_clean_2024[candidate_vars].copy()
    
    # Apply log transform to household_num (as done during training)
    if "household_num" in X_test.columns:
        X_test["household_num"] = np.log1p(X_test["household_num"])
        X_test = X_test.rename(columns={"household_num": "log_household_num"})
    
    X_matrix = sm.add_constant(pd.concat([df_clean_2024[fixed_vars], X_test], axis=1))
    X_matrix = X_matrix[model_m3.params.index] # Ensure correct column order

    # 3. Predict & Back-transform
    y_pred_real = np.expm1(model_m3.predict(X_matrix)).clip(lower=0)
    y_true_real = df_clean_2024[y_col]

    # 4. Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan
    
    test_metrics_m3.append({
        "Fuel": fuel,
        "Test R2 (2024)": r2_score(y_true_real, y_pred_real),
        "RMSE": rmse,
        "NRMSE": nrmse,
        "MAE": mean_absolute_error(y_true_real, y_pred_real)
    })

df_val_m2 = pd.DataFrame(test_metrics_m3).set_index("Fuel")
print("=== VALIDATION: Model (2019-23) Results on 2024 Data ===")
display(df_val_m2.round(3))

### 4.3 Validating Model 3 (trained on 2019–2023, with trend) on 2024

Trend terms are evaluated at their 2024 values (τ = 5, τ² = 25),
matching the assumption used later in the forecast (paper Section 3.5).


In [ ]:
# ==========================================
# VALIDATION: Model (2019-23+Trend) tested on 2024 (FIXED)
# ==========================================
test_metrics_m5 = []

for fuel in fuel_lists:
    y_col = f"new_{fuel}"
    # Skip if the model was not fitted for this fuel type
    if y_col not in df_test_2024.columns or fuel not in results_summary_m3:
        continue
    
    # 1. Retrieve the trained model (paper Model 3)
    res = results_summary_m3[fuel]
    model_m3 = res["model"]
    
    # 2. Clean and prepare the 2024 feature matrix
    df_clean_2024 = df_test_2024[[y_col] + candidate_vars + fixed_vars].dropna()
    
    # 3. Build the predictor matrix X_test
    X_test = df_clean_2024[candidate_vars].copy()
    
    # Add trend terms: tau = 5 (2019 = 0 -> 2024 = 5), tau^2 = 25
    X_test['trend'] = 5
    X_test['trend_sq'] = 25
    
    # Log-transform household count and rename to match the training-time column
    if "household_num" in X_test.columns:
        X_test["log_household_num"] = np.log1p(X_test["household_num"])
        X_test = X_test.drop(columns=["household_num"]) # drop the original column

    # 4. Concatenate fixed (dummy) and continuous predictors
    X_combined = pd.concat([df_clean_2024[fixed_vars], X_test], axis=1)
    
    # Explicitly add the intercept column the model expects
    X_matrix = sm.add_constant(X_combined, has_constant='add')
    
    # 5. Reorder columns to match the training-time parameter order
    # (model_m3.params.index includes 'const', so column order must include it)
    try:
        X_matrix = X_matrix[model_m3.params.index]
    except KeyError as e:
        print(f"Error for {fuel}: {e}")
        print(f"Required columns: {model_m3.params.index.tolist()}")
        print(f"Available columns: {X_matrix.columns.tolist()}")
        continue

    # 6. Predict and back-transform to the real (un-logged) scale
    y_pred_real = np.expm1(model_m3.predict(X_matrix)).clip(lower=0)
    y_true_real = df_clean_2024[y_col]

    # 7. Compute evaluation metrics
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    y_range = y_true_real.max() - y_true_real.min()
    nrmse = rmse / y_range if y_range > 0 else np.nan
    
    test_metrics_m5.append({
        "Fuel": fuel,
        "Test R2 (2024)": r2_score(y_true_real, y_pred_real),
        "RMSE": rmse,
        "NRMSE": nrmse,
        "MAE": mean_absolute_error(y_true_real, y_pred_real)
    })

# Display results
df_val_m3 = pd.DataFrame(test_metrics_m5).set_index("Fuel")
print("=== VALIDATION: Model (2019-23+Trend) Results on 2024 Data ===")
display(df_val_m3.round(3))

In [ ]:
# Reproduces paper Table 4.3 — out-of-sample R^2, MAE, NRMSE across
# all three paper models for each fuel type.
import pandas as pd

# ==========================================
# CONSOLIDATED VALIDATION TABLE (TESTED ON 2024)
# ==========================================

# 1. Collect the per-model validation DataFrames into a single dict
#    with consistent display labels
val_frames = {
    "(1) Model (2023)": df_validation_2024, # Trained on 2023 only
    "(2) Model (2019-23)": df_val_m2,       # Trained on 2019-2023, no trend
    "(3) Model (2019-23+Trend)": df_val_m3  # Trained on 2019-2023, with quadratic trend
}

# 2. Concatenate vertically into a master frame
all_val_data = []
for model_name, frame in val_frames.items():
    temp = frame.copy().reset_index()
    temp['Model'] = model_name
    all_val_data.append(temp)

df_val_master = pd.concat(all_val_data)

# 3. Pivot for side-by-side comparison
# Metrics shown: Test R2 (2024), MAE, NRMSE
df_val_comparison = df_val_master.pivot(index="Fuel", columns="Model")

# Specify the metric order
metrics_to_show = ["Test R2 (2024)", "MAE", "NRMSE"]
fuel_order = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids", "Total"]

df_val_comparison = df_val_comparison.reindex(columns=metrics_to_show, level=0)
df_val_comparison = df_val_comparison.reindex(fuel_order)

print("=== 2024 VALIDATION COMPARISON MATRIX (Out-of-Sample Test) ===")
display(df_val_comparison.round(3))

# --- LaTeX export ---
# print(df_val_comparison.round(3).to_latex(multirow=True, multicolumn=True))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score

# ==========================================
# DIAGNOSTIC PLOT: ACTUAL VS PREDICTED (2024)
# ==========================================

def plot_hybrid_diagnostic(target_fuel):
    """
    Compare Model 1, 2, and 3 predictions for 2024 to identify accuracy drops.
    """
    y_col = f"new_{target_fuel}"
    
    # 1. Build the 2024 test frame (or reuse if already in scope)
    # Build df_test_2024 locally if it isn't already defined
    if 'df_test_2024' not in locals():
        df_24 = df[df["year"] == 2024].copy()
        X_cat_24 = pd.get_dummies(df_24["Gruppkod"], prefix="Gruppkod", dtype=float)
        df_test_2024_local = pd.concat([df_24.reset_index(drop=True), X_cat_24.reset_index(drop=True)], axis=1)
        for col in fixed_vars:  # fixed_vars is captured from training
            if col not in df_test_2024_local.columns: df_test_2024_local[col] = 0.0
    else:
        df_test_2024_local = df_test_2024.copy()

    # Drop rows with missing predictors
    df_clean_24 = df_test_2024_local.dropna(subset=[y_col] + candidate_vars).copy()
    y_actual = df_clean_24[y_col]
    
    # 2. Generate predictions for each model
    preds = {}

    # --- (1) Model (2023): results_summary_m1 ---
    m1 = results_summary_m1[target_fuel]["model"]
    X1 = df_clean_24.copy()
    X1["log_household_num"] = np.log1p(X1["household_num"])
    X1_mat = sm.add_constant(X1, has_constant='add')[m1.params.index]
    preds["(1) Model (2023)"] = np.expm1(m1.predict(X1_mat)).clip(lower=0)

    # --- (2) Model (2019-23): results_summary_m2 ---
    m2 = results_summary_m2[target_fuel]["model"]
    X2 = df_clean_24.copy()
    X2["log_household_num"] = np.log1p(X2["household_num"])
    X2_mat = sm.add_constant(X2, has_constant='add')[m2.params.index]
    preds["(2) Model (2019-23)"] = np.expm1(m2.predict(X2_mat)).clip(lower=0)

    # --- (3) Model (2019-23+Trend): results_summary_m3 ---
    m3 = results_summary_m3[target_fuel]["model"]
    X3 = df_clean_24.copy()
    X3["log_household_num"] = np.log1p(X3["household_num"])
    X3['trend'] = 5      # tau = year - 2019; 2024 -> 5
    X3['trend_sq'] = 25  # 5^2
    X3_mat = sm.add_constant(X3, has_constant='add')[m3.params.index]
    preds["(3) Model (2019-23+Trend)"] = np.expm1(m3.predict(X3_mat)).clip(lower=0)

    # 3. Plot
    fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharey=True, sharex=True)
    model_labels = list(preds.keys())
    
    for i, label in enumerate(model_labels):
        ax = axes[i]
        y_pred = preds[label]
        r2 = r2_score(y_actual, y_pred)
        
        # Scatter (transparent teal: city-size disparity is wide)
        sns.scatterplot(x=y_actual, y=y_pred, alpha=0.5, ax=ax, color='teal', edgecolor='w')
        
        # 45-degree line (perfect prediction)
        max_val = max(y_actual.max(), y_pred.max())
        ax.plot([0.1, max_val], [0.1, max_val], 'r--', lw=2)
        
        # Log scale exposes deviations for small municipalities
        ax.set_xscale('log')
        ax.set_yscale('log')
        
        ax.set_title(f"{label}\nTest R2: {r2:.3f}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Actual Registrations (2024)")
        if i == 0: ax.set_ylabel("Predicted Registrations (2024)")
        ax.grid(True, which="both", ls="--", alpha=0.5)

    plt.suptitle(f"Diagnostic Scatter Plot: Actual vs Predicted (2024) for {target_fuel}", fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()

# --- Execute on the two hybrid fuel types where Model 3 underperforms ---
plot_hybrid_diagnostic("Electric-hybrids")
plot_hybrid_diagnostic("Plug-in-hybrids")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score

def plot_hybrid_diagnostic_zoom(target_fuel, limit=500):
    """
    Plot 2024 predictions from paper Models 1, 2, 3 on a real (linear) scale, restricted to municipalities below `limit` registrations.
    """
    y_col = f"new_{target_fuel}"
    
    # 1. Prepare the 2024 frame
    df_clean_24 = df_test_2024.dropna(subset=[y_col] + candidate_vars).copy()
    y_actual_real = df_clean_24[y_col]
    
    preds_real = {}

    # --- Paper Model 1 ---
    m1 = results_summary_m1[target_fuel]["model"]
    X1 = df_clean_24.copy()
    X1["log_household_num"] = np.log1p(X1["household_num"])
    X1_mat = sm.add_constant(X1, has_constant='add')[m1.params.index]
    preds_real["(1) Model (2023)"] = np.expm1(m1.predict(X1_mat)).clip(lower=0)

    # --- Paper Model 2 ---
    m2 = results_summary_m2[target_fuel]["model"]
    X2 = df_clean_24.copy()
    X2["log_household_num"] = np.log1p(X2["household_num"])
    X2_mat = sm.add_constant(X2, has_constant='add')[m2.params.index]
    preds_real["(2) Model (2019-23)"] = np.expm1(m2.predict(X2_mat)).clip(lower=0)

    # --- Paper Model 3 ---
    m3 = results_summary_m3[target_fuel]["model"]
    X3 = df_clean_24.copy()
    X3["log_household_num"] = np.log1p(X3["household_num"])
    X3['trend'] = 5
    X3['trend_sq'] = 25 
    X3_mat = sm.add_constant(X3, has_constant='add')[m3.params.index]
    preds_real["(3) Model (2019-23+Trend)"] = np.expm1(m3.predict(X3_mat)).clip(lower=0)

    # 3. Plot subset (limit = 500 by default)
    fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharey=True, sharex=True)
    model_labels = ["(1) Model (2023)", "(2) Model (2019-23)", "(3) Model (2019-23+Trend)"]
    
    for i, label in enumerate(model_labels):
        ax = axes[i]
        y_pred = preds_real[label]
        
        # Subset: only municipalities below `limit` actual registrations
        mask = (y_actual_real <= limit)
        y_act_sub = y_actual_real[mask]
        y_pred_sub = y_pred[mask]
        
        # R^2 computed on the subset
        r2_sub = r2_score(y_act_sub, y_pred_sub)
        
        # Scatter
        sns.scatterplot(x=y_act_sub, y=y_pred_sub, alpha=0.6, ax=ax, color='teal', edgecolor='w')
        
        # 45-degree line
        ax.plot([0, limit], [0, limit], 'r--', lw=2, label='Perfect Prediction')
        
        # Axis limits
        ax.set_xlim(0, limit)
        ax.set_ylim(0, limit)
        
        ax.set_title(f"{label}\n(Subset R2: {r2_sub:.3f})", fontsize=14, fontweight='bold')
        ax.set_xlabel("Actual Registrations (0-500)")
        if i == 0: ax.set_ylabel("Predicted Registrations (0-500)")
        ax.grid(True, ls="--", alpha=0.5)

    plt.suptitle(f"Zoomed Diagnostic (0-{limit} units): {target_fuel} in 2024", fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()

# --- Execute ---
plot_hybrid_diagnostic_zoom("Electric-hybrids", limit=1000)
plot_hybrid_diagnostic_zoom("Plug-in-hybrids", limit=1000)


## 5. Forecast 2025–2040

Project new registrations with Model 3 (the chosen forecasting model),
then combine with empirical scrapping rates, fuel-type VKT, and emission
factors to produce annual CO₂ trajectories for every kommun.


In [ ]:
# Working subset of df with all variables needed by the forecast loop:
# stock and new-registration counts for the five fuel types plus Total,
# plus the six predictors and the Gruppkod label.
df_forecast = df[['Kommun-kod', 'Municipality', 'Petrol', 'Diesel', 'Electricity',
       'Electric-hybrids', 'Plug-in-hybrids', 'Total', 'Gruppkod', 'new_Petrol', 
       'new_Diesel', 'new_Electricity', 'new_Electric-hybrids', 'new_Plug-in-hybrids', 
       'new_Total', 'year', 'median_household_income', 'bachelor_plus_pct', 
       'household_num', 'household_num_growth', 'Avg_Price_öre_kWh', 'corporation_percentage']]

In [ ]:
# For each predictor that will be rolled forward over 2025-2040, compute
# both a CAGR (compound annual growth rate, anchored at 2019 and 2024)
# and an average annual growth rate (mean of year-on-year pct_change).
# CAGR is the default in get_predicted_path; average is kept for sensitivity.
# ==========================================
# 1. SETUP & DATA FILTERING
# ==========================================
# Use data from 2019 to 2024 to calculate historical trends
df_trend = df_forecast.sort_values(['Kommun-kod', 'year'])

# Variables for which we need to calculate growth rates
target_vars = ['median_household_income', 'bachelor_plus_pct', 'household_num', 'corporation_percentage']

# ==========================================
# 2. CALCULATE CAGR (Compound Annual Growth Rate)
# Formula: ((V_final / V_begin) ** (1 / t)) - 1
# ==========================================
def calculate_cagr(group, variables):
    start_val = group[group['year'] == 2019].set_index('Kommun-kod')[variables]
    end_val = group[group['year'] == 2024].set_index('Kommun-kod')[variables]
    n_years = 2024 - 2019
    
    # Calculate CAGR
    cagr = (end_val / start_val) ** (1 / n_years) - 1
    return cagr

# ==========================================
# 3. CALCULATE AVERAGE ANNUAL GROWTH RATE
# Formula: Mean of year-over-year percentage changes
# ==========================================
def calculate_avg_growth(group, variables):
    # Calculate year-over-year growth for each variable within the group
    yoy_growth = group.set_index('year')[variables].pct_change()
    # Return the mean growth rate across available years
    return yoy_growth.mean()

# Execute calculations per municipality
# We'll create a summary dataframe for growth rates
cagr_per_mun = calculate_cagr(df_trend, target_vars).reset_index()
avg_growth_per_mun = df_trend.groupby('Kommun-kod').apply(lambda x: calculate_avg_growth(x, target_vars)).reset_index()

# ==========================================
# 4. PREPARE GROWTH SUMMARY TABLE
# ==========================================
# Rename columns to distinguish between CAGR and Average
cagr_per_mun.columns = ['Kommun-kod'] + [f"{v}_cagr" for v in target_vars]
avg_growth_per_mun.columns = ['Kommun-kod'] + [f"{v}_avg" for v in target_vars]

# Merge results
growth_rates_summary = pd.merge(cagr_per_mun, avg_growth_per_mun, on='Kommun-kod')

# Add household_num_growth substitute (using household_num_cagr)
# This will be used as a constant value for the growth rate column in the future
growth_rates_summary['household_num_growth_future'] = growth_rates_summary['household_num_cagr']
#growth_rates_summary['household_num_growth_future'] = growth_rates_summary['household_num_avg]
growth_rates_summary['Avg_Price_öre_kWh_future'] = 2.0


# Display first few rows
print("=== Historical Growth Rates per Municipality (2019-2024) ===")
display(growth_rates_summary.head())

In [ ]:
def get_predicted_path(kommun_kod, df_hist, growth_summary, models_dict, method='cagr'):
    """
    Generate a continuous path of predicted values (Hindcast + Forecast) from 2019 to 2040.
    Includes stabilization logic: Trend terms are frozen at 2024 levels for the forecast period.
    """
    # 1. PREPARE PREDICTORS FOR ENTIRE TIMELINE (2019-2040)
    mun_data_hist = df_hist[df_hist['Kommun-kod'] == kommun_kod].sort_values('year').copy()
    if mun_data_hist.empty:
        return f"Error: Kommun-kod {kommun_kod} not found."
    
    rates = growth_summary[growth_summary['Kommun-kod'] == kommun_kod].iloc[0]
    base_2024 = mun_data_hist[mun_data_hist['year'] == 2024].iloc[0].copy()
    
    future_rows = []
    current_vals = base_2024.copy()
    
    for year in range(2025, 2041):
        row = current_vals.copy()
        row['year'] = year
        for var in ['median_household_income', 'bachelor_plus_pct', 'household_num', 'corporation_percentage']:
            row[var] = current_vals[var] * (1 + rates[f"{var}_{method}"])
        row['Avg_Price_öre_kWh'] = current_vals['Avg_Price_öre_kWh'] * 1.02
        row['household_num_growth'] = rates[f"household_num_{method}"]
        future_rows.append(row)
        current_vals = row

    df_full_predictors = pd.concat([mun_data_hist, pd.DataFrame(future_rows)], ignore_index=True)

    # 2. RUN MODEL PREDICTIONS (2019-2040)
    X_categorical = pd.get_dummies(df_full_predictors["Gruppkod"], prefix="Gruppkod", dtype=float)
    for col in fixed_vars: 
        if col not in X_categorical.columns:
            X_categorical[col] = 0.0
            
    df_full_ready = pd.concat([df_full_predictors.reset_index(drop=True), X_categorical.reset_index(drop=True)], axis=1)

    # --- STABILIZATION LOGIC START ---
    # Calculate trend from baseline 2019
    df_full_ready['trend'] = df_full_ready['year'] - 2019
    df_full_ready['trend_sq'] = df_full_ready['trend'] ** 2
    
    # FREEZE trend at 2024 level (t=5, t^2=25) for any year after 2024
    # This prevents the quadratic function from exploding or dropping to zero unrealistically
    df_full_ready.loc[df_full_ready['year'] > 2024, 'trend'] = 5
    df_full_ready.loc[df_full_ready['year'] > 2024, 'trend_sq'] = 25
    # --- STABILIZATION LOGIC END ---

    output_table = pd.DataFrame({"year": df_full_ready["year"]})
    output_table["Status"] = output_table["year"].apply(lambda x: "Hindcast" if x <= 2024 else "Forecast")

    for fuel in fuel_lists:
        if fuel not in models_dict: continue
        m_info = models_dict[fuel]
        model = m_info["model"]
        
        X_input = df_full_ready.copy()
        if "log_household_num" in model.params.index:
            X_input["log_household_num"] = np.log1p(X_input["household_num"])
            
        X_matrix = sm.add_constant(X_input, has_constant='add')[model.params.index]
        
        log_pred = model.predict(X_matrix)
        output_table[f"pred_new_{fuel}"] = np.expm1(log_pred).clip(lower=0)

    return output_table

### 5.1 Empirical scrapping rates

Scrapping is recovered from the stock identity (paper Eq. 3.3):
$$\text{scrapped}_{X,t} = \text{stock}_{X,t-1} + \text{new}_{X,t} - \text{stock}_{X,t}$$
Negative values from registry revisions are clipped to zero.


In [ ]:
# 1. Sort by municipality and year
df_scrapped = df_forecast.sort_values(['Kommun-kod', 'year']).copy()

# Fuel list (including Total)
fuel_types = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids", "Total"]

for fuel in fuel_types:
    # Previous-year stock (stock at t-1)
    prev_stock = df_scrapped.groupby('Kommun-kod')[fuel].shift(1)
    
    # New registrations at t
    new_reg = df_scrapped[f"new_{fuel}"]
    
    # Current stock (stock at t)
    current_stock = df_scrapped[fuel]
    
    # Scrapped count (also absorbs out-migration and exports)
    # scrapped = stock(t-1) + new(t) - stock(t)
    scrapped_col = f"scrapped_{fuel}"
    df_scrapped[scrapped_col] = prev_stock + new_reg - current_stock
    
    # Scrapping rate = scrapped / stock at t-1
    # Pandas yields NaN on divide-by-zero, which is fine here
    df_scrapped[f"scr_rate_{fuel}"] = df_scrapped[scrapped_col] / prev_stock
    
    # Registry noise can leave the scrap count negative; clip to 0
    # (e.g. when used-car imports exceed scrappage)
    df_scrapped[scrapped_col] = df_scrapped[scrapped_col].clip(lower=0)
    df_scrapped[f"scr_rate_{fuel}"] = df_scrapped[f"scr_rate_{fuel}"].clip(lower=0)

# 2. Quick sanity-check display
print("=== Scrapped Rate Calculation (Example: Stockholm) ===")
view_cols = ['year', 'Municipality', 'Total', 'new_Total', 'scrapped_Total', 'scr_rate_Total']
view_cols = ['year', 'Municipality', 'Total', 'new_Total', 'scrapped_Total', 'scr_rate_Total']

display(df_scrapped[view_cols].round(4))


In [ ]:
# Average scrapping rate per fuel type within each SKR group, averaged
# across all years and municipalities in the group (paper Figure 4.2).
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. AGGREGATE BY MUNICIPALITY CLASSIFICATION (Gruppkod)
# ==========================================

# List of columns to average (Scrapped Rates)
rate_cols = [
    'scr_rate_Petrol', 'scr_rate_Diesel', 'scr_rate_Electricity', 
    'scr_rate_Electric-hybrids', 'scr_rate_Plug-in-hybrids'
]

# Calculate the mean rate for each classification group
# This averages across all years (2019-2024) and all municipalities in that group
df_group_scrap = df_scrapped.groupby('Gruppkod')[rate_cols].mean()

# Rename columns for the plot legend to be cleaner
df_group_scrap.columns = ["Petrol", "Diesel", "Electricity", "E-hybrid", "PHEV"]

# ==========================================
# 2. VISUALIZATION
# ==========================================
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

# Plotting as a grouped bar chart
df_group_scrap.plot(kind='bar', figsize=(14, 7), width=0.8, alpha=0.8)

# Chart Styling
plt.title("Average Scrapped Rate by Municipality Classification (2019-2024)", fontsize=16)
plt.ylabel("Average Scrapped Rate (Ratio of Total Stock)", fontsize=12)
plt.xlabel("Kommun Classification (Gruppkod)", fontsize=12)
plt.xticks(rotation=0) # Keep labels horizontal for readability
plt.legend(title="Fuel Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ==========================================
# 3. DISPLAY THE NUMERICAL TABLE
# ==========================================
print("=== Average Scrapped Rates per Kommun Group (Numerical Table) ===")
display(df_group_scrap.round(4))

### 5.2 Build the master 2019–2040 panel


In [ ]:
# ==========================================
# 1. Per-municipality, per-fuel mean scrapping rate
# ==========================================
fuel_types = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]

# Compute the mean scrapping rate over 2019-2024 per Kommun-kod
avg_scrap_rates = df_scrapped.groupby('Kommun-kod')[[f"scr_rate_{f}" for f in fuel_types]].mean()

# Fill NA (e.g. municipality never held that fuel type) with 0
avg_scrap_rates = avg_scrap_rates.fillna(0)

In [ ]:
# ==========================================
# 2. Build the 290 kommun x 22 year master panel
# ==========================================
final_dataset_list = []
fuel_types = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]

# All municipality codes
all_kommun_kods = df_forecast['Kommun-kod'].unique()

for kod in all_kommun_kods:
    # --- A. Actual values 2019-2024 ---
    actual_part = df_forecast[df_forecast['Kommun-kod'] == kod].sort_values('year').copy()
    actual_part['Status'] = 'Actual'
    # Compute scrapped counts for the historical period
    for f in fuel_types:
        prev_s = actual_part[f].shift(1)
        actual_part[f"scrapped_{f}"] = (prev_s + actual_part[f"new_{f}"] - actual_part[f]).clip(lower=0)
    
    # --- B. Predicted new registrations 2019-2040 ---
    # NB: get_predicted_path freezes trend=5, trend_sq=25 from 2025 onward
    # If you patch that function, check that predicted_path does not explode
    predicted_path = get_predicted_path(kod, df_forecast, growth_rates_summary, results_summary_m3, method='cagr')
    
    # --- C. 2025-2040 simulation ---
    # End-of-2024 stock = starting point for the simulation
    last_stock = actual_part[actual_part['year'] == 2024][fuel_types].iloc[0].to_dict()
    current_stock = last_stock.copy()
    
    # Per-municipality average scrapping rates
    city_rates = avg_scrap_rates.loc[kod].to_dict()
    
    future_rows = []
    for year in range(2025, 2041):
        # Pull the predicted new registrations for this year from predicted_path
        pred_row = predicted_path[predicted_path['year'] == year].iloc[0]
        
        new_year_data = {
            'Kommun-kod': kod,
            'Municipality': actual_part['Municipality'].iloc[0],
            'Gruppkod': actual_part['Gruppkod'].iloc[0],
            'year': year,
            'Status': 'Forecast'
        }
        
        total_stock_this_year = 0
        for f in fuel_types:
            # 1. Predicted new registrations this year
            new_val = pred_row[f"pred_new_{f}"]
            new_year_data[f"new_{f}"] = new_val
            
            # 2. Scrapped this year = previous stock x historical mean rate
            rate = city_rates[f"scr_rate_{f}"]
            scrapped_val = current_stock[f] * rate
            new_year_data[f"scrapped_{f}"] = scrapped_val # record the scrapped count
            
            # 3. End-of-year stock: stock_t = stock_{t-1} + new - scrapped
            updated_stock = current_stock[f] + new_val - scrapped_val
            updated_stock = max(0, updated_stock) # floor at zero
            
            new_year_data[f] = updated_stock
            current_stock[f] = updated_stock # carry into next year
            total_stock_this_year += updated_stock
            
        new_year_data['Total'] = total_stock_this_year
        future_rows.append(new_year_data)
    
    # Concatenate this kommun's Actual and Forecast segments
    df_city_full = pd.concat([actual_part, pd.DataFrame(future_rows)], ignore_index=True)
    final_dataset_list.append(df_city_full)

# Combine every kommun into one master table
df_master_19_40 = pd.concat(final_dataset_list, ignore_index=True)

print("Master dataset built (includes scrapped counts).")

In [ ]:
# Trim the column set for a readable display
cols = ['Kommun-kod', 'Municipality', 'year', 'Status', 'Petrol', 'Diesel', 
        'Electricity', 'Electric-hybrids','Plug-in-hybrids', 'Total']

display(df_master_19_40[df_master_19_40['Municipality'] == 'Stockholm'][cols].head(25).round(0))

# Sanity-check the row count (290 kommuner x 22 years = 6380)
print(f"Total rows: {len(df_master_19_40)}")

### 5.3 Vehicle kilometres travelled (VKT)

VKT is taken from the Trafikanalys 2024 national report and held constant
over the forecast horizon (paper Section 3.5). Caveat: holding VKT fixed
ignores travel-behaviour changes that may accompany electrification.


In [ ]:
# VKT by fuel type from the 2024 Trafikanalys road-traffic statistics.
# Fuel categories that don't enter the model (e.g. ethanol, NG) are
# dropped via the mapping in the emission-factor section below.
import pandas as pd

file_path = DATA_DIR / "vkt_data_2024.xlsx"
sheet_name = 'vkt_2024'

vkt_prepped = pd.read_excel(file_path, sheet_name=sheet_name)
vkt_prepped = vkt_prepped.round(0)
vkt_prepped


### 5.4 CO₂ emission factors from EEA 2023 registrations

Per-fuel emission factors (g/km) are derived from the EEA 2023 new-
registration dataset for Sweden, taking the mean WLTP value within each
(fuel type, fuel mode) combination relevant to the model.


In [ ]:
file_path = DATA_DIR / "EEA_2023_SE_data.csv"

# Columns we need from the EEA wide table
needed_cols = [
    "Ft",           # Fuel type
    "Fm",           # Fuel mode (M, H, P, E)
    "Ewltp (g/km)", # WLTP CO2 in g/km
    "year"          # Year
]

# usecols keeps the read cheap on this wide file
df_emission = pd.read_csv(file_path, usecols=needed_cols)

# Quick check
print(df_emission.columns)
display(df_emission.head())

In [ ]:
# Display unique pairs of Ft and Fm
unique_pairs = df_emission[['Ft', 'Fm']].drop_duplicates()
print(unique_pairs)

Filter to the fuel categories used in this study: Petrol, Diesel, EVs,
Electric-hybrids, and Plug-in-hybrids; drop everything else (NG, E85,
hydrogen, etc.).


In [ ]:
# Create a cross-tabulation table
cross_tab = pd.crosstab(df_emission['Ft'], df_emission['Fm'])
display(cross_tab)

In [ ]:
import pandas as pd

# 1. Ensure Ewltp is numeric
df_emission['Ewltp (g/km)'] = pd.to_numeric(df_emission['Ewltp (g/km)'], errors='coerce')

# 2. Pivot table
# Rows: Ft, columns: Fm, values: mean Ewltp
emission_matrix = df_emission.pivot_table(
    index='Ft', 
    columns='Fm', 
    values='Ewltp (g/km)', 
    aggfunc='mean'
)

# 3. Round for readability
emission_matrix_rounded = emission_matrix.round(1)

# 4. Display
print("=== Mean CO2 emissions (Ewltp g/km) by Fuel type (Ft) and Mode (Fm) ===")
display(emission_matrix_rounded)

In [ ]:
import pandas as pd

# 1. (Ft, Fm) -> classification mapping
mapping = {
    ('petrol', 'M'): 'Petrol',
    ('diesel', 'M'): 'Diesel',
    ('electric', 'E'): 'EVs',
    ('petrol', 'H'): 'Electric-hybrids',
    ('diesel', 'H'): 'Electric-hybrids',
    ('petrol/electric', 'P'): 'Plug-in-hybrids',
    ('diesel/electric', 'P'): 'Plug-in-hybrids'
}

# 2. Apply the mapping row-by-row
def classify_vehicle(row):
    return mapping.get((row['Ft'], row['Fm']), None)

# 3. Assign the new classification column
df_emission['classification'] = df_emission.apply(classify_vehicle, axis=1)

# 4. Drop rows that didn't match (e.g. NG, E85, hydrogen)
df_emission_cleaned = df_emission.dropna(subset=['classification']).copy()

# 5. Sanity check
print("=== Unique (Ft, Fm) pairs after filtering, with classification ===")
print(df_emission_cleaned.groupby(['classification', 'Ft', 'Fm']).size())

# Row counts before/after
print(f"\nRemaining rows: {len(df_emission_cleaned)} (Original: {len(df_emission)})")

In [ ]:
import pandas as pd

# 1. Cast to numeric
df_emission['Ewltp (g/km)'] = pd.to_numeric(df_emission['Ewltp (g/km)'], errors='coerce')

# 2. (Ft, Fm) -> classification mapping
mapping = {
    ('petrol', 'M'): 'Petrol',
    ('diesel', 'M'): 'Diesel',
    ('electric', 'E'): 'EVs',
    ('petrol', 'H'): 'Electric-hybrids',
    ('diesel', 'H'): 'Electric-hybrids',
    ('petrol/electric', 'P'): 'Plug-in-hybrids',
    ('diesel/electric', 'P'): 'Plug-in-hybrids'
}

# 3. Apply the mapping
df_emission['Ft_lower'] = df_emission['Ft'].str.lower()
df_emission['Fm_upper'] = df_emission['Fm'].str.upper()
df_emission['classification'] = df_emission.apply(lambda x: mapping.get((x['Ft_lower'], x['Fm_upper'])), axis=1)

# 4. Aggregate
detailed_table = df_emission.dropna(subset=['classification']).groupby(['classification', 'Ft_lower', 'Fm_upper']).agg(
    Count=('Ewltp (g/km)', 'count'),
    Avg_Ewltp_gkm=('Ewltp (g/km)', 'mean')
).reset_index()

# 5. Enforce a custom display order on the classification axis
# Desired order
fuel_order = ["Petrol", "Diesel", "EVs", "Electric-hybrids", "Plug-in-hybrids"]

# Cast to ordered categorical
detailed_table['classification'] = pd.Categorical(
    detailed_table['classification'], 
    categories=fuel_order, 
    ordered=True
)

# Sort by the ordered categorical
detailed_table = detailed_table.sort_values(['classification', 'Ft_lower']).reset_index(drop=True)

# Cosmetic column renames
detailed_table = detailed_table.rename(columns={'Ft_lower': 'Ft', 'Fm_upper': 'Fm'})

# 6. Display
print("=== Detailed Breakdown: Ordered by Custom List ===")
display(detailed_table.round(2))

In [ ]:
import numpy as np

# 1. Cast Ewltp to numeric (NaN for unparseable cells)
df_emission['Ewltp (g/km)'] = pd.to_numeric(df_emission['Ewltp (g/km)'], errors='coerce')

# 2. (Ft, Fm) -> classification mapping; 'electric' renamed to 'Electricity'
mapping = {
    ('petrol', 'M'): 'Petrol',
    ('diesel', 'M'): 'Diesel',
    ('electric', 'E'): 'Electricity',  # match the registration column
    ('petrol', 'H'): 'Electric-hybrids',
    ('diesel', 'H'): 'Electric-hybrids',
    ('petrol/electric', 'P'): 'Plug-in-hybrids',
    ('diesel/electric', 'P'): 'Plug-in-hybrids'
}

# 3. Apply the mapping
# Lowercase Ft, uppercase Fm to make matching robust
df_emission['Ft'] = df_emission['Ft'].str.lower()
df_emission['Fm'] = df_emission['Fm'].str.upper() # (Fm is a single letter: M, H, P, E)

df_emission['classification'] = df_emission.apply(lambda x: mapping.get((x['Ft'], x['Fm'])), axis=1)

# 4. Keep only the five classification categories
df_cleaned = df_emission.dropna(subset=['classification']).copy()

# 5. Per-class mean Ewltp (= registration-weighted, since each row is one car)
factor_prepped = df_cleaned.groupby('classification').agg(
    Sample_Size=('Ewltp (g/km)', 'count'),
    Avg_gCO2_per_km=('Ewltp (g/km)', 'mean')
).reset_index()

# 6. Enforce the display order
fuel_order = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]
factor_prepped['classification'] = pd.Categorical(factor_prepped['classification'], categories=fuel_order, ordered=True)
factor_prepped = factor_prepped.sort_values('classification')

print("=== Fuel-type emission factors from Sweden 2023 new registrations ===")
display(factor_prepped.round(2))

### 5.5 Combine stocks, VKT and emission factors

Annual CO₂ emissions per fuel type = stock × VKT × emission factor,
applied row-wise to df_master_19_40.


In [ ]:
# ==========================================
# 1. Prepare VKT and emission-factor lookups
# ==========================================

# Build a fuel -> VKT (km) lookup
# Rename 'Electric' -> 'Electricity' to match the registration columns
vkt_dict = vkt_prepped.set_index('fuel_type')['vkt_km'].to_dict()
if 'Electric' in vkt_dict:
    vkt_dict['Electricity'] = vkt_dict.pop('Electric')

# Emission factor (gCO2/km) lookup from factor_prepped
em_factor_dict = factor_prepped.set_index('classification')['Avg_gCO2_per_km'].to_dict()

# Target fuels
target_fuels = ["Petrol", "Diesel", "Electricity", "Electric-hybrids", "Plug-in-hybrids"]

# ==========================================
# 2. Compute annual emissions for 2019-2040
# ==========================================
# Formula: annual emissions (g) = stock * annual VKT (km) * emission factor (g/km)

for fuel in target_fuels:
    vkt = vkt_dict.get(fuel, 0)
    factor = em_factor_dict.get(fuel, 0)
    
    # Annual emissions per fuel (grams)
    df_master_19_40[f"emissions_{fuel}"] = (
        df_master_19_40[fuel] * vkt * factor
    )

# Sum across fuels
emission_cols = [f"emissions_{f}" for f in target_fuels]
df_master_19_40["total_emissions_g"] = df_master_19_40[emission_cols].sum(axis=1)

# Convert to metric tonnes for plotting
df_master_19_40["total_emissions_ton"] = df_master_19_40["total_emissions_g"] / 1_000_000

print("Emission calculation complete.")
df_master_19_40.head()

In [ ]:
# Identify, for each SKR group, the municipality whose 2023 median income
# and household count is closest (in standardised Euclidean distance) to
# the within-group medians. These nine kommuner feed paper Figure 4.3.
import matplotlib.pyplot as plt

# ==========================================
# Step 1: Pick a representative municipality for each SKR group
# ==========================================
groups = ["A1", "A2", "B3", "B4", "B5", "C6", "C7", "C8", "C9"]

group_labels = {
    "A1": "A1: Large cities",
    "A2": "A2: Commuter (large city)",
    "B3": "B3: Larger cities",
    "B4": "B4: Commuter (larger city)",
    "B5": "B5: Low commuting",
    "C6": "C6: Smaller cities",
    "C7": "C7: Commuter (smaller city)",
    "C8": "C8: Rural",
    "C9": "C9: Rural (Tourism)"
}

# Use 2023 data to pick the representative
df_2023 = df[df["year"] == 2023].copy()

representative_municipalities = {}

for group in groups:
    df_g = df_2023[df_2023["Gruppkod"] == group].copy()
    
    # Within-group medians
    median_income = df_g["median_household_income"].median()
    median_households = df_g["household_num"].median()
    
    # Standardise income and households, then pick the closest kommun
    # in standardised Euclidean distance
    income_std = df_g["median_household_income"].std()
    hh_std = df_g["household_num"].std()
    
    df_g["distance"] = (
        ((df_g["median_household_income"] - median_income) / income_std) ** 2 +
        ((df_g["household_num"] - median_households) / hh_std) ** 2
    ) ** 0.5
    
    rep_muni = df_g.loc[df_g["distance"].idxmin(), "Municipality"]
    representative_municipalities[group] = rep_muni
    print(f"{group_labels[group]}: {rep_muni}")

print("\nRepresentative municipalities:", representative_municipalities)

In [ ]:
# ==========================================
# Step 2: Emissions plot for the nine representative kommuner (3x3 grid)
# ==========================================

colors_map = {
    "Petrol": "red",
    "Diesel": "black",
    "Electricity": "green",
    "Electric-hybrids": "purple",
    "Plug-in-hybrids": "blue"
}
fuels = ["Petrol", "Diesel", "Electric-hybrids", "Plug-in-hybrids"]

fig, axes = plt.subplots(3, 3, figsize=(24, 20))
axes_flat = axes.flatten()

all_handles = []
all_labels  = []

for idx, group in enumerate(groups):
    ax = axes_flat[idx]
    city_name = representative_municipalities[group]
    
    city_data = df_master_19_40[
        df_master_19_40["Municipality"] == city_name
    ].sort_values("year")
    
    # Stacked area
    y_stack    = [city_data[f"emissions_{f}"] / 1e6 for f in fuels]
    color_list = [colors_map[f] for f in fuels]
    
    ax.stackplot(city_data["year"], y_stack,
                 labels=fuels, colors=color_list, alpha=0.7)
    
    # Total emissions overlay
    ax.plot(city_data["year"], city_data["total_emissions_ton"],
            label="Total", color="darkorange", linewidth=2.5, linestyle="--")
    
    # Boundary between historical and forecast
    ax.axvline(x=2024.5, color="gray", linestyle=":", alpha=0.8, linewidth=1.5)
    
    # Title (one line, slightly larger)
    ax.set_title(f"{group_labels[group]}: {city_name}",
                 fontsize=16, fontweight="bold", pad=10)
    
    ax.set_xlabel("Year", fontsize=15)
    ax.set_ylabel("Annual CO$_2$ (Mt)", fontsize=13)
    ax.set_xticks(range(2019, 2041, 4))
    ax.set_xlim(2019, 2040)
    ax.tick_params(labelsize=12)
    ax.grid(True, linestyle=":", alpha=0.4)
    
    # Grab legend handles from the first panel
    if idx == 0:
        handles, labels = ax.get_legend_handles_labels()
        all_handles = list(reversed(handles))
        all_labels  = list(reversed(labels))

fig.legend(
    all_handles, all_labels,
    loc            = "lower center",
    ncol           = len(all_labels),
    fontsize       = 16,
    title          = "Fuel Type / Emission Source",
    title_fontsize = 14,
    frameon        = True,
    bbox_to_anchor = (0.5, 0.01)
)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(OUTPUT_DIR / "emissions_9municipalities.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# Step 3: Emit a standalone .tex file (paper Table 4.7)
# ==========================================

lines = []

# ---- Preamble ----
lines.append(r"\documentclass[a4paper,10pt]{article}")
lines.append(r"\usepackage[utf8]{inputenc}")
lines.append(r"\usepackage[T1]{fontenc}")
lines.append(r"\usepackage{booktabs}")
lines.append(r"\usepackage{geometry}")
lines.append(r"\geometry{margin=2cm, landscape}")   # landscape so all columns fit
lines.append(r"\usepackage{siunitx}")               # decimal alignment
lines.append(r"\sisetup{group-separator={,}, group-minimum-digits=4}")
lines.append(r"\begin{document}")
lines.append(r"")

# ---- Table body ----
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Actual and forecast vehicle registrations by municipality and year}")
lines.append(r"\label{tab:table9}")
lines.append(r"\small")
# siunitx S columns for decimal alignment (4 numeric columns + Total)
lines.append(r"\begin{tabular}{llcc S[table-format=9.3] S[table-format=9.3] S[table-format=9.3] S[table-format=9.3] S[table-format=9.3]}")
lines.append(r"\toprule")
lines.append(
    r"Group & Municipality & Year & Status & "
    r"{Petrol} & {Diesel} & {Electric-hybrids} & {Plug-in-hybrids} & {Total (tCO$_2$)} \\"
)
lines.append(r"\midrule")

for group in groups:
    city_name = representative_municipalities[group]
    
    for i, year in enumerate([2019, 2024, 2030, 2040]):
        row_data = df_master_19_40[
            (df_master_19_40["Municipality"] == city_name) &
            (df_master_19_40["year"] == year)
        ]
        if len(row_data) == 0:
            continue
        
        row    = row_data.iloc[0]
        status = "Actual" if year <= 2024 else "Forecast"
        
        group_disp = group     if i == 0 else ""
        city_disp  = city_name if i == 0 else ""
        city_esc   = city_disp.replace("&", r"\&").replace("_", r"\_")
        
        # siunitx S columns expect raw numbers
        petrol = f"{row['emissions_Petrol']:.3f}"
        diesel = f"{row['emissions_Diesel']:.3f}"
        eh     = f"{row['emissions_Electric-hybrids']:.3f}"
        ph     = f"{row['emissions_Plug-in-hybrids']:.3f}"
        total  = f"{row['total_emissions_ton']:.3f}"
        
        line = (
            f"{group_disp} & {city_esc} & {year} & {status} & "
            f"{petrol} & {diesel} & {eh} & {ph} & {total} \\\\"
        )
        lines.append(line)
    
    lines.append(r"\midrule")

lines[-1] = r"\bottomrule"

lines.append(r"\end{tabular}")
lines.append(r"\end{table}")
lines.append(r"")
lines.append(r"\end{document}")

# ---- Save the .tex file ----
tex_str  = "\n".join(lines)
tex_path = OUTPUT_DIR / "table9_representative_municipalities.tex"

with open(tex_path, "w", encoding="utf-8") as f:
    f.write(tex_str)

print(f"Saved: {tex_path}")
print("-" * 60)
print(tex_str)